# W4.1 — Graph utilities và MessagePassing

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** Toy graph, utils/link_graph.py; không cần Cora.

Prototype chiếu h_j=W x_j rồi cộng theo node nhận i. `forward` chuẩn bị h,
`message` lấy h_j, `aggregate` cộng bằng index_add. Không có attention, chưa phải GAT.
Nguồn: [MessagePassing](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.MessagePassing.html).

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
import numpy as np
import torch
from utils.experiment import (
    read_json,
    write_json,
    seed_everything,
    array_hash,
    mapping_hash,
    environment,
    new_run,
    publish_result,
    expect_error,
)
from utils.link_graph import (
    validate_edges,
    canonicalize,
    bidirectional,
    with_self_loops,
    neighborhoods,
    pair_set,
    load_cora,
    make_split,
    audit_split,
)


D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/training_demo.json')
print(config)

{'dataset': 'toy', 'model': 'linear_regression', 'task': 'regression', 'seed': 42, 'device': 'cpu', 'learning_rate': 0.05, 'weight_decay': 0.0, 'max_epochs': 150, 'patience': 12, 'monitor': 'validation_mse', 'mode': 'min', 'min_delta': 1e-06}


In [ ]:
from torch_geometric.nn import MessagePassing

from layers.projected_sum import ProjectedSum


In [ ]:
seed_everything(config['seed'])
checks = {}
e = np.array([[1,0,1,2,2],[0,1,0,1,2]],dtype=np.int64)
canon = canonicalize(e,4)
assert np.array_equal(canon,np.array([[0,1],[1,2]]))
assert np.array_equal(canonicalize(canon,4),canon)
adj = bidirectional(e,4)
assert np.array_equal(bidirectional(adj,4),adj)
loops = with_self_loops(adj,4)
assert np.array_equal(with_self_loops(loops,4),loops)
assert np.array_equal(loops[:,loops[0]==loops[1]],np.tile(np.arange(4),(2,1)))
assert len(neighborhoods(adj,4)) == 4 and neighborhoods(adj,4)[3].size == 0
checks['canonical_bidirectional_loops_isolated'] = {'status':'passed','canonical':canon.tolist(),'loop_count':4}
for name,bad in [('shape',np.array([0,1])),('dtype',np.array([[0.],[1.]])),('range',np.array([[0],[4]])),('negative',np.array([[-1],[0]]))]:
    checks['reject_'+name] = expect_error(lambda bad=bad:validate_edges(bad,4))
x = torch.tensor([[1.,0.],[0.,1.],[1.,1.],[2.,2.]],requires_grad=True)
edge = torch.tensor([[0,2,1],[1,1,2]],dtype=torch.long)
layer = ProjectedSum(2,2)
with torch.no_grad(): layer.projection.weight.copy_(torch.eye(2))
out = layer(x,edge)
manual = torch.tensor([[0.,0.],[2.,1.],[0.,1.],[0.,0.]])
torch.testing.assert_close(out,manual)
reference = torch.zeros_like(out)
for u,v in edge.T.tolist(): reference[v] = reference[v]+layer.projection(x[u])
torch.testing.assert_close(out,reference)
torch.testing.assert_close(layer(x,edge[:,torch.tensor([2,0,1])]),out)
out.square().sum().backward()
assert torch.isfinite(x.grad).all() and x.grad.abs().sum()>0
assert torch.isfinite(layer.projection.weight.grad).all() and layer.projection.weight.grad.abs().sum()>0
torch.testing.assert_close(layer(x,torch.empty((2,0),dtype=torch.long)),torch.zeros_like(out))
checks['message_passing_reference_gradient_permutation'] = {'status':'passed','output':out.detach().tolist(),'shape':list(out.shape),'dtype':str(out.dtype),'device':str(out.device)}
run = new_run(PROJECT_ROOT,'w4_graph',config)
result = publish_result(PROJECT_ROOT,run,'results/week04/graph_checks.json',{'status':'passed','checks':checks},config,4,'W4.1','projected_sum','toy')
print(checks); print('run:',run.name)

{'canonical_bidirectional_loops_isolated': {'status': 'passed', 'canonical': [[0, 1], [1, 2]], 'loop_count': 4}, 'reject_shape': {'status': 'passed', 'observed': 'edge_index phải có shape [2,M], dtype integer'}, 'reject_dtype': {'status': 'passed', 'observed': 'edge_index phải có shape [2,M], dtype integer'}, 'reject_range': {'status': 'passed', 'observed': 'Node index ngoài miền [0,num_nodes)'}, 'reject_negative': {'status': 'passed', 'observed': 'Node index ngoài miền [0,num_nodes)'}, 'message_passing_reference_gradient_permutation': {'status': 'passed', 'output': [[0.0, 0.0], [2.0, 1.0], [0.0, 1.0], [0.0, 0.0]], 'shape': [4, 2], 'dtype': 'torch.float32', 'device': 'cpu'}}
run: 20260928T070150_w4_graph_8e183432


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Source và target nằm ở hàng nào?
2. Vì sao cần num_nodes khi node cuối cô lập?
3. Prototype thiếu bước nào để trở thành GAT?